In [2]:
import os
import json
import ast
import re
import sys
import uuid
from typing import List
import shutil
import sqlite3

import inspect

from pathlib import Path
import importlib.util

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# from sentence_transformers import SentenceTransformer

In [3]:
# 1) Add the *project root* (the parent of "src") to sys.path
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
sys.path.append(src_path)

# 2) Import from the package path
from b_func_prompt_texts import * 

## Create DB with NZZ chart data

In [4]:
df_chart_table = pd.read_csv("../data/db_tables/df_chart_table.csv")


df_model_table = pd.read_csv("../data/db_tables/df_model_table.csv")


df_data_table = pd.read_csv("../data/db_tables/df_data_table.csv")


df_event_table = pd.read_csv("../data/db_tables/df_event_table.csv")


df_chart_type_table = pd.read_csv("../data/db_tables/df_chart_type_table.csv")

In [5]:
# # SQLite-Verbindung erstellen
# conn = sqlite3.connect("chart_database.db")
# cursor = conn.cursor()

# Tabellen zuerst löschen, falls vorhanden
# cursor.execute("DROP TABLE IF EXISTS chart")
# cursor.execute("DROP TABLE IF EXISTS data_event")
# cursor.execute("DROP TABLE IF EXISTS chart_data")
# cursor.execute("DROP TABLE IF EXISTS chart_type")
# cursor.execute("DROP TABLE IF EXISTS model")
# cursor.execute("DROP TABLE IF EXISTS generation_run")
# cursor.execute("DROP TABLE IF EXISTS alt_text_embedding")
# cursor.execute("DROP TABLE IF EXISTS alt_text")

In [6]:
# # 1. Tabelle: model
# cursor.execute("""
# CREATE TABLE IF NOT EXISTS model (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     company TEXT,
#     model_series TEXT,
#     URL TEXT
# )
# """)

# # 2. Tabelle: chart_type
# cursor.execute("""
# CREATE TABLE IF NOT EXISTS chart_type (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     type TEXT
# )
# """)

# # 3. Tabelle: chart
# cursor.execute("""
# CREATE TABLE IF NOT EXISTS chart (
#     id TEXT PRIMARY KEY ,
#     chart_type_id INTEGER,
#     x_axis_type TEXT,
#     y_axis_type TEXT,
#     x_axis_min FLOAT,
#     x_axis_max FLOAT,
#     y_axis_min FLOAT,
#     y_axis_max FLOAT,
#     complex BOOL,
#     title TEXT,
#     subtitle TEXT,
#     notes TEXT,
#     FOREIGN KEY(chart_type_id) REFERENCES chart_type(id)              
# )
# """)

# # 4. Tabelle: chart_data
# cursor.execute("""
# CREATE TABLE IF NOT EXISTS chart_data (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     chart_id TEXT,
#     x_value TEXT,
#     x_category TEXT,
#     y_value TEXT,
#     y_category TEXT,
#     row_index INTEGER,
#     col_index INTEGER,
#     prognosis BOOL,
#     highlighted BOOL,
#     FOREIGN KEY(chart_id) REFERENCES chart(id)
# )
# """)

# # 5. Tabelle: data_event
# cursor.execute("""
# CREATE TABLE IF NOT EXISTS data_event (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     chart_id TEXT,
#     type TEXT,
#     date TEXT,
#     label TEXT,
#     FOREIGN KEY(chart_id) REFERENCES chart(id)
# )
# """)



# # Schreibe die Daten aus den DataFrames in die Tabellen
# df_chart_type_table.drop(columns=["Unnamed: 0"], errors="ignore").to_sql("chart_type", conn, if_exists="append", index=False)
# df_model_table.drop(columns=["Unnamed: 0", "created"], errors="ignore").to_sql("model", conn, if_exists="append", index=False)
# df_chart_table.drop(columns=["Unnamed: 0", "chart_type"], errors="ignore").to_sql("chart", conn, if_exists="append", index=False)
# df_data_table.drop(columns=["Unnamed: 0"], errors="ignore").to_sql("chart_data", conn, if_exists="append", index=False)
# df_event_table.drop(columns=["Unnamed: 0"], errors="ignore").to_sql("data_event", conn, if_exists="append", index=False)


# # Commit & schließen
# conn.commit()
# conn.close()

### DB überprüfen

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

# Tabellenliste
dbs = ["model", "chart_type", "chart", "chart_data", "data_event"] #"gold_summary"

for db in dbs:
    print(f"\n------ Tabelle: {db} ------")
    cursor.execute(f"SELECT * FROM {db} LIMIT 3;")
    data = cursor.fetchall()

    # Spaltennamen anzeigen
    column_names = [description[0] for description in cursor.description]
    print(" | ".join(column_names))  # schöne Formatierung

    # Daten anzeigen
    for row in data:
        print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()


------ Tabelle: model ------
id | company | model_series | URL
1 | Meta | meta-llama/llama-4-maverick:free | https://openrouter.ai/meta-llama/llama-4-maverick:free
2 | google | google/gemini-2.5-flash-preview-09-2025 | https://statics.teams.cdn.office.net/evergreen-assets/safelinks/2/atp-safelinks.html
3 | OpenAI | openai/o4-mini | https://openrouter.ai/openai/o4-mini

------ Tabelle: chart_type ------
id | type
0 | Bar
1 | Line
2 | StackedBar

------ Tabelle: chart ------
id | chart_type_id | x_axis_type | y_axis_type | x_axis_min | x_axis_max | y_axis_min | y_axis_max | complex | title | subtitle | notes
0023e8fed9d111fed753bb3f6b0afe78 | 0 | numeric | numeric | 2012.0 | 2017.0 | 5.01155899 | 20.79092879 | 1 | Auch der Handel mit der EU mit Autoteilen ist aus Sicht der USA defizitär | in Mrd. $ | None
0057a266a9c555637dabccece11c5468 | 0 | numeric | numeric | 1998.0 | 2019.0 | 1500000.0 | 2500000.0 | 0 | Besucher am Züri-Fäscht | None | None
005e2925d4b679defb5afa4daef05ee5 | 0 | ca

## Insert prompt texts into DB

In [ ]:
# # Verbindung zur SQLite-Datenbank (Datei wird erstellt, falls nicht vorhanden)
# conn = sqlite3.connect("../chart_database.db")
# cursor = conn.cursor()

# # Fremdschlüssel-Unterstützung aktivieren (wichtig in SQLite)
# cursor.execute("PRAGMA foreign_keys = ON;")

# #Tabellen zuerst löschen (in Abhängigkeitsreihenfolge: evaluation_run → alt_text → generation_run → prompt_text/model/metric)
# cursor.execute("DROP TABLE IF EXISTS prompt_text_function")

# # Tabelle: prompt_text_function
# cursor.execute("""
# CREATE TABLE IF NOT EXISTS prompt_text_function (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     name TEXT,
#     content TEXT
# );
# """)

# # Änderungen speichern und Verbindung schließen
# conn.commit()
# conn.close()


# # Korrekte Datei
# file_path = Path(Path.cwd().parent / "src" / "b_func_prompt_texts.py")

# if not file_path.exists():
#     raise FileNotFoundError(f"Datei nicht gefunden: {file_path}")

# # Modul dynamisch importieren
# module_name = "b_prompt_texts"
# spec = importlib.util.spec_from_file_location(module_name, file_path)
# module = importlib.util.module_from_spec(spec)
# spec.loader.exec_module(module)


# # SQLite vorbereiten
# conn = sqlite3.connect("../chart_database.db")
# cursor = conn.cursor()

# # Alle Funktionen im Modul finden und in die DB schreiben
# functions = inspect.getmembers(module, inspect.isfunction)

# for func_name, func_obj in functions:
#     try:
#         source = inspect.getsource(func_obj)
#     except OSError:
#         print("Error, function not found")
#         continue

#     cursor.execute(
#         "INSERT INTO prompt_text_function (name, content) VALUES (?, ?)",
#         (func_name, source)
#     )

# # Speichern und schließen
# conn.commit()
# conn.close()

#### Prompt_text funktionen updaten

In [ ]:
# import sqlite3
# import importlib.util
# import inspect
# from pathlib import Path

# DB_PATH = "../chart_database.db"

# FILE_PATH = Path(Path.cwd().parent / "src" / "b_func_prompt_texts.py")
# MODULE_NAME = "b_func_prompt_texts"

# def load_module_from_path(file_path, module_name):
#     spec = importlib.util.spec_from_file_location(module_name, file_path)
#     module = importlib.util.module_from_spec(spec)
#     spec.loader.exec_module(module)
#     return module

# def update_prompt_text_function_table(conn, module):
#     cur = conn.cursor()

#     # 1) Vorhandene IDs in der Tabelle abrufen
#     cur.execute("SELECT id FROM prompt_text_function ORDER BY id ASC")
#     existing_ids = [row[0] for row in cur.fetchall()]

#     # 2) Funktionen aus Modul abrufen
#     functions = [
#         (name, obj)
#         for name, obj in inspect.getmembers(module, inspect.isfunction)
#         if obj.__module__ == MODULE_NAME
#     ]

#     # Sortierung damit Zuordnung stabil bleibt
#     functions.sort(key=lambda x: x[0])

#     if len(existing_ids) != len(functions):
#         raise ValueError(
#             f"Anzahl Funktionen ({len(functions)}) ≠ Anzahl Tabellenzeilen ({len(existing_ids)})."
#         )

#     # 3) Inhalte überschreiben
#     for (func_name, func_obj), row_id in zip(functions, existing_ids):
#         source = inspect.getsource(func_obj)

#         cur.execute("""
#             UPDATE prompt_text_function
#             SET name = ?, content = ?
#             WHERE id = ?
#         """, (func_name, source, row_id))

#     conn.commit()
#     cur.close()
#     print(f"{len(existing_ids)} Einträge erfolgreich aktualisiert.")

# def main():
#     module = load_module_from_path(FILE_PATH, MODULE_NAME)
#     conn = sqlite3.connect(DB_PATH)
#     update_prompt_text_function_table(conn, module)
#     conn.close()

# if __name__ == "__main__":
#     main()


### DB überprüfen

In [ ]:
# Check if insert was successfull
conn = sqlite3.connect("../chart_database.db")

query_prompt_text_function = "SELECT * FROM prompt_text_function"

df_prompt_text_function = pd.read_sql_query(query_prompt_text_function, conn)

conn.close()

df_prompt_text_function

,id,name,content
0,1,alt_prompt_text_1,"def alt_prompt_text_1(chart_info, csv_path, pn..."
1,2,prompt_faktenkorrektheit,"def prompt_faktenkorrektheit(alt_text: str, cs..."
2,3,prompt_klarheit,"def prompt_klarheit(alt_text: str, csv_text: s..."
3,4,prompt_kuerze,"def prompt_kuerze(alt_text: str, csv_text: str..."
4,5,prompt_neutralität,"def prompt_neutralität(alt_text: str, csv_text..."
5,6,prompt_vollstaendigkeit,"def prompt_vollstaendigkeit(alt_text: str, csv..."
6,7,prompt_wahrgenommene_vollstaendigkeit,def prompt_wahrgenommene_vollstaendigkeit(alt_...
7,8,prompt_faktenkorrektheit_reason,def prompt_faktenkorrektheit_reason(alt_text: ...
8,9,prompt_klarheit_reason,"def prompt_klarheit_reason(alt_text: str, csv_..."
9,10,prompt_kuerze_reason,"def prompt_kuerze_reason(alt_text: str, csv_te..."


## Create Generation-Pipeline Tables

In [ ]:
# # generation_run, alt_text_embedding, alt_text
# conn = sqlite3.connect("../chart_database.db")

# # Tabelle: generation_run
# conn.execute("""
# CREATE TABLE IF NOT EXISTS generation_run (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     model_id INTEGER,
#     prompt_text_function_id INTEGER,
#     created_at TEXT,
#     temperature FLOAT,
#     n_variants INTEGER,
#     FOREIGN KEY (model_id) REFERENCES model(id),
#     FOREIGN KEY (prompt_text_function_id) REFERENCES prompt_text_function(id)
# );"""
# )


# # Tabelle: alt_text_embedding 
# conn.execute("""
# CREATE TABLE IF NOT EXISTS alt_text_embedding (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     alt_text_id INTEGER,
#     model_id INTEGER,
#     dim INTEGER,
#     normalized INTEGER,
#     embedding BLOB,
#     FOREIGN KEY (alt_text_id) REFERENCES alt_text(id),
#     FOREIGN KEY (model_id) REFERENCES model(id)
# );
# """)



# # Tabelle: alt_text
# conn.execute("""
# CREATE TABLE IF NOT EXISTS alt_text (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     chart_id INTEGER,
#     generation_run_id INTEGER,
#     variant_no INTEGER,
#     short_description_metadata TEXT,
#     short_description_overview TEXT,
#     long_description TEXT,
#     FOREIGN KEY (generation_run_id) REFERENCES generation_run(id),
#     FOREIGN KEY (chart_id) REFERENCES chart(id));
# """)

# conn.close()

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

# Tabellenliste
dbs = ["generation_run", "alt_text_embedding", "alt_text"]

for db in dbs:
    print(f"\n------ Tabelle: {db} ------")
    cursor.execute(f"SELECT * FROM {db} LIMIT 3;")
    data = cursor.fetchall()

    # Spaltennamen anzeigen
    column_names = [description[0] for description in cursor.description]
    print(" | ".join(column_names))  # schöne Formatierung

    # Daten anzeigen
    for row in data:
        print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()


------ Tabelle: generation_run ------
id | model_id | prompt_text_function_id | created_at | temperature | n_variants
1 | 2 | 1 | 2025-11-16T10:27:51.513631 | 1.0 | 1
2 | 2 | 1 | 2025-12-11T13:33:54.230344 | 0.4 | 10
4 | 2 | 1 | 2025-12-11T14:32:39.834946 | 1.0 | 10

------ Tabelle: alt_text_embedding ------
id | alt_text_id | model_id | dim | normalized | embedding | metadata_embedding | overview_embedding | long_description_embedding | normalized_parts
27 | 1 | 4 | 384 | 1 | b'\xce\x81\x85\xbd\x97\xb1q=\xdc\xb85\xbc\x12\x06\xc6;\x18\xf3`\xbdQ"\x0e<\x11\x80\x90\xbd\xc7\xc1\xd3<\xe1\xf8\x13\xbd\xac\x07U\xbd\x05 9<\x0b\xa4\x92\xbd\xbdn#=\xcc2\x8d\xbc+\x8a\xb7\xbc\x9d4y\xbcP_ <\x1a:u\xbd\xae\xa2\xfb\xbc\xb0\xca[=R\xe4\x9e;wi\x89\xbd\x03\xdd\xa4\xbcH\xc96\xbd)\xd7\x8c=\x9cP\x8c\xbc}\xeb\x9e\xbd\x15\xdbL=\'t~\xbb3\xc8\xa1\xbc\xa8cO=\'^\xfb=\xaf!W\xbdy"\x13\xbc\x0f\xb5\xfb=y\x7f)\xbd\xafz\xf9< \xcb\x9c\xbd\xd1\xebh\xbc\x8e\x16\x1a=\n\x1e\xcc;Fl\x91\xbcu\x1a\xc6\xbd\xaaP\xf4;\xa2"v\xbc7<\xa

## Create LLM-Pipeline Tables

In [ ]:
# # Tabelle: llm_evaluation
# conn = sqlite3.connect("../chart_database.db")

# conn.execute("""
# CREATE TABLE IF NOT EXISTS llm_evaluation (
    # id INTEGER PRIMARY KEY AUTOINCREMENT,
    # evaluation_run_id INTEGER,
    # alt_text_id INTEGER,
    # no_eval INTEGER,

    # score_clarity INTEGER,
    # reasoning_clarity TEXT,

    # score_completeness INTEGER,
    # reasoning_completeness TEXT,

    # score_conciseness INTEGER,
    # reasoning_conciseness TEXT,

    # score_preceived_completeness INTEGER,
    # reasoning_preceived_completeness TEXT,

    # score_neutrality INTEGER,
    # reasoning_neutrality TEXT,

    # score_correctness INTEGER,
    # reasoning_correctness TEXT,
             
    # FOREIGN KEY (evaluation_run_id) REFERENCES evaluation_run(id)
# );
# """)


# # Tabelle: metric
# conn.execute("""
# CREATE TABLE IF NOT EXISTS metric (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     alt_text_id INTEGER NOT NULL,
#     tokens_short_description_metadata INTEGER,
#     tokens_short_description_overview INTEGER,
#     tokens_long_description INTEGER,
#     FOREIGN KEY (alt_text_id) REFERENCES alt_text(id)
# );
# """)


# conn = sqlite3.connect("../chart_database.db")

# # Tabelle: evaluation_run
# conn.execute("""
# CREATE TABLE IF NOT EXISTS evaluation_run (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     generation_run_id INTEGER NOT NULL,
#     model_id INTEGER NOT NULL,
#     num_evals INTEGER NOT NULL,
#     func_clarity_id INTEGER,
#     func_completeness_id INTEGER,
#     func_conciseness_id INTEGER,
#     func_preceived_completeness_id INTEGER,
#     func_neutrality_id INTEGER,
#     func_correctness_id INTEGER,      
#     FOREIGN KEY (generation_run_id) REFERENCES generation_run(id),
#     FOREIGN KEY (model_id) REFERENCES model(id),
#     FOREIGN KEY (func_clarity_id) REFERENCES prompt_text_function(id),
#     FOREIGN KEY (func_completeness_id) REFERENCES prompt_text_function(id),
#     FOREIGN KEY (func_conciseness_id) REFERENCES prompt_text_function(id),
#     FOREIGN KEY (func_preceived_completeness_id) REFERENCES prompt_text_function(id),
#     FOREIGN KEY (func_neutrality_id) REFERENCES prompt_text_function(id),
#     FOREIGN KEY (func_correctness_id) REFERENCES prompt_text_function(id)
# );
# """)

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

# Tabellenliste
dbs = ["llm_evaluation", "metric", "evaluation_run"]

for db in dbs:
    print(f"\n------ Tabelle: {db} ------")
    cursor.execute(f"SELECT * FROM {db} LIMIT 3;")
    data = cursor.fetchall()

    # Spaltennamen anzeigen
    column_names = [description[0] for description in cursor.description]
    print(" | ".join(column_names))  # schöne Formatierung

    # Daten anzeigen
    for row in data:
        print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()


------ Tabelle: llm_evaluation ------
id | evaluation_run_id | alt_text_id | no_eval | score_clarity | reasoning_clarity | score_completeness | reasoning_completeness | score_conciseness | reasoning_conciseness | score_preceived_completeness | reasoning_preceived_completeness | score_neutrality | reasoning_neutrality | score_correctness | reasoning_correctness
49 | 5 | 1 | 0 | 5 | None | 3 | None | 2 | None | 5 | None | 2 | None | 3 | None
50 | 5 | 3 | 0 | 5 | None | 4 | None | 2 | None | 5 | None | 3 | None | 5 | None
51 | 5 | 5 | 0 | 2 | None | 3 | None | 1 | None | 4 | None | 3 | None | 2 | None

------ Tabelle: metric ------
id | alt_text_id | tokens_short_description_metadata | tokens_short_description_overview | tokens_long_description
25 | 1 | 206 | 256 | 382
26 | 3 | 299 | 140 | 233
27 | 5 | 396 | 289 | 555

------ Tabelle: evaluation_run ------
id | generation_run_id | model_id | num_evals | func_clarity_id | func_completeness_id | func_conciseness_id | func_preceived_complet

In [ ]:
# # SQLite-Verbindung erstellen
# conn = sqlite3.connect("../chart_database.db")
# cursor = conn.cursor()

# cursor.execute("DROP TABLE IF EXISTS llm_evaluation ")
# cursor.execute("DROP TABLE IF EXISTS metric ")
# cursor.execute("DROP TABLE IF EXISTS evaluation_run ")

# # Verbindung schließen
# conn.close()

## Insert gold-standard texts (mit embeddings) into DB

In [ ]:
# # SQLite-Verbindung erstellen
# conn = sqlite3.connect("../chart_database.db")
# cursor = conn.cursor()

# cursor.execute("DROP TABLE IF EXISTS gold_standard_alt_text ")

# # Verbindung schließen
# conn.close()

In [17]:
DB_PATH = "chart_database.db"
RAW_PATH = "gold_standard_alt_texts_raw.txt"

In [18]:
# import sqlite3

# def migrate_gold_standard_alt_text_schema(conn: sqlite3.Connection, default_judge_model_id: int):
#     """
#     Migriert gold_standard_alt_text auf das gewünschte Schema:
#     - baut neue Tabelle mit allen Spalten/Constraints
#     - kopiert bestehende Daten rüber
#     - setzt judge_model_id auf default_judge_model_id (weil NOT NULL)
#     """
#     cur = conn.cursor()

#     # FK enforcement an (optional, aber sinnvoll)
#     cur.execute("PRAGMA foreign_keys = ON;")

#     # Prüfen ob Tabelle existiert
#     row = cur.execute("""
#         SELECT name FROM sqlite_master WHERE type='table' AND name='gold_standard_alt_text'
#     """).fetchone()
#     if not row:
#         raise ValueError("Tabelle gold_standard_alt_text existiert nicht.")

#     # Neue Tabelle erstellen
#     cur.executescript("""
#     CREATE TABLE IF NOT EXISTS gold_standard_alt_text_new (
#         id INTEGER PRIMARY KEY AUTOINCREMENT,
#         chart_id TEXT NOT NULL,

#         short_description_metadata TEXT,
#         short_description_overview TEXT,
#         long_description TEXT,

#         embedding BLOB,
#         embedding_model_id INTEGER,

#         tokens_short_description_metadata INTEGER,
#         tokens_short_description_overview INTEGER,
#         tokens_long_description INTEGER,

#         judge_model_id INTEGER NOT NULL,
#         func_clarity_id INTEGER,
#         func_completeness_id INTEGER,
#         func_conciseness_id INTEGER,
#         func_preceived_completeness_id INTEGER,
#         func_neutrality_id INTEGER,
#         func_correctness_id INTEGER,

#         score_clarity INTEGER,
#         reason_clarity TEXT,
#         score_completeness INTEGER,
#         reason_completeness TEXT,
#         score_conciseness INTEGER,
#         reason_conciseness TEXT,
#         score_preceived_completeness INTEGER,
#         reason_preceived_completeness TEXT,
#         score_neutrality INTEGER,
#         reason_neutrality TEXT,
#         score_correctness INTEGER,
#         reason_correctness TEXT,

#         FOREIGN KEY (embedding_model_id) REFERENCES model(id),
#         FOREIGN KEY (chart_id) REFERENCES chart(id),

#         FOREIGN KEY (judge_model_id) REFERENCES model(id),
#         FOREIGN KEY (func_clarity_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_completeness_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_conciseness_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_preceived_completeness_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_neutrality_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_correctness_id) REFERENCES prompt_text_function(id)
#     );
#     """)

#     # Daten kopieren: alles was existiert + judge_model_id mit default setzen
#     cur.execute("""
#         INSERT INTO gold_standard_alt_text_new (
#             id,
#             chart_id,
#             short_description_metadata,
#             short_description_overview,
#             long_description,
#             embedding,
#             embedding_model_id,
#             judge_model_id
#         )
#         SELECT
#             id,
#             chart_id,
#             short_description_metadata,
#             short_description_overview,
#             long_description,
#             embedding,
#             embedding_model_id,
#             ?
#         FROM gold_standard_alt_text
#     """, (default_judge_model_id,))

#     # Alte Tabelle ersetzen
#     cur.executescript("""
#         DROP TABLE gold_standard_alt_text;
#         ALTER TABLE gold_standard_alt_text_new RENAME TO gold_standard_alt_text;
#     """)

#     conn.commit()
#     cur.close()


In [19]:
# conn = sqlite3.connect(DB_PATH)
# migrate_gold_standard_alt_text_schema(conn, default_judge_model_id=3)
# conn.close()


In [20]:
# conn = sqlite3.connect(DB_PATH)

# # 3 Tabelle: gold_standard_alt_text
# conn.executescript("""
#     CREATE TABLE IF NOT EXISTS gold_standard_alt_text (
#         id INTEGER PRIMARY KEY AUTOINCREMENT,
#         chart_id TEXT NOT NULL,
                   
#         short_description_metadata TEXT,
#         short_description_overview TEXT,
#         long_description TEXT,
                   
#         embedding BLOBB,
#         embedding_model_id INTEGER,
                   
#         tokens_short_description_metadata INTEGER,
#         tokens_short_description_overview INTEGER,
#         tokens_long_description INTEGER,
                   
#         judge_model_id INTEGER NOT NULL,
#         func_clarity_id INTEGER,
#         func_completeness_id INTEGER,
#         func_conciseness_id INTEGER,
#         func_preceived_completeness_id INTEGER,
#         func_neutrality_id INTEGER,
#         func_correctness_id INTEGER,    

#         score_clarity INTEGER,
#         score_completeness INTEGER,
#         score_conciseness INTEGER,
#         score_preceived_completeness INTEGER,
#         score_neutrality INTEGER,
#         score_correctness INTEGER,
                
#         FOREIGN KEY (embedding_model_id) REFERENCES model(id),
#         FOREIGN KEY (chart_id) REFERENCES chart(id),
                   
#         FOREIGN KEY (judge_model_id) REFERENCES model(id),
#         FOREIGN KEY (func_clarity_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_completeness_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_conciseness_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_preceived_completeness_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_neutrality_id) REFERENCES prompt_text_function(id),
#         FOREIGN KEY (func_correctness_id) REFERENCES prompt_text_function(id)
#     );
# """)

In [21]:
# MODEL_SERIES  = "sentence-transformers/all-MiniLM-L6-v2"
# BATCH_SIZE    = 64

# # ================================
# # Deine globale DB-Verbindung
# # ================================
# CONN = sqlite3.connect("chart_database.db")
# cursor = CONN.cursor()
# cursor.execute("PRAGMA foreign_keys = ON;")


# # ================================
# # 1) GOLD TEXT PARSEN 
# # ================================
# raw = Path(RAW_PATH).read_text(encoding="utf-8")

# pattern = re.compile(
#     r'ID:\s*"?(?P<chart_id>[0-9a-f]+)"?\s*'
#     r'Kurzbeschreibung:\s*(?P<short_meta>.*?)'
#     r'Überblick:\s*(?P<overview>.*?)'
#     r'Lange Beschreibung:\s*(?P<long>.*?)(?=\n.*ID:|\Z)',
#     re.S,
# )

# records = []
# for m in pattern.finditer(raw):
#     records.append(
#         {
#             "chart_id": m.group("chart_id").strip(),
#             "short_description_metadata": m.group("short_meta").strip(),
#             "short_description_overview": m.group("overview").strip(),
#             "long_description": m.group("long").strip(),
#         }
#     )

# print(f"Parsed {len(records)} records")
# if records:
#     from pprint import pprint
#     pprint(records[0])

In [22]:
# # ================================
# # 2) Embedding Model ID holen
# # ================================
# cursor.execute(
#     "SELECT id FROM model WHERE model_series=?;",
#     (MODEL_SERIES,)
# )

# embedding_model_id = cursor.fetchone()
# embedding_model_id = int(embedding_model_id[0])
# print(f"Using embedding model ID: {embedding_model_id}")


# # ================================
# # 3) SBERT EMBEDDINGS ERZEUGEN
# # ================================
# def concat_text(rec):
#     return (
#         f"[META] {rec.get('short_description_metadata','').strip()}\n"
#         f"[OVERVIEW] {rec.get('short_description_overview','').strip()}\n"
#         f"[LONG] {rec.get('long_description','').strip()}"
#     ).strip()

# texts = [concat_text(r) for r in records]

# model = SentenceTransformer(MODEL_SERIES)

# emb_list = []
# for start in range(0, len(texts), BATCH_SIZE):
#     batch = texts[start:start+BATCH_SIZE]
#     embs = model.encode(batch, convert_to_numpy=True, normalize_embeddings=False)
#     emb_list.extend(list(embs.astype(np.float32)))

# assert len(emb_list) == len(records)


# # ================================
# # 4) RECORDS mit EMBEDDINGS füllen
# # ================================
# for rec, emb in zip(records, emb_list):
#     rec["embedding"] = emb.tobytes()
#     rec["embedding_model_id"] = embedding_model_id


# # ================================
# # 5) UPSERT in gold_standard_alt_text
# # ================================
# cursor.executemany(
#     """
#     INSERT INTO gold_standard_alt_text (
#         chart_id,
#         short_description_metadata,
#         short_description_overview,
#         long_description,
#         embedding,
#         embedding_model_id
#     )
#     VALUES (
#         :chart_id,
#         :short_description_metadata,
#         :short_description_overview,
#         :long_description,
#         :embedding,
#         :embedding_model_id
#     )
#     """,
#     records,
# )

# CONN.commit()
# print("Gold standards inserted + embeddings stored.")


## Create und füllen der people-evaluation Table

In [23]:
# import sqlite3

# conn = sqlite3.connect(DB_PATH)
# cur = conn.cursor()

# kriterien_lst = ['score_neutrality', 'score_clarity', 'score_conciseness', 'score_preceived_completeness']

# # score_per_chart[alt_text_id]: {person_name: [k1, k2, k3, k4]}
# scores_per_chart = {
#     1:  {'Person A':[5,5,5,5],'Person B':[5,3,5,5],'Person C':[5,5,5,5],'Person D':[5,4,4,5],'Person E':[5,4,5,3]},
#     3:  {'Person A':[5,5,5,5],'Person B':[5,4,5,4],'Person C':[5,4,5,5],'Person D':[5,4,5,4],'Person E':[5,5,5,4]},
#     5:  {'Person A':[5,5,5,5],'Person B':[5,5,5,5],'Person C':[5,2,5,5],'Person D':[5,4,5,4],'Person E':[5,4,4,4]},
#     7:  {'Person A':[5,4,5,5],'Person B':[5,5,5,5],'Person C':[5,5,5,5],'Person D':[5,3,5,5],'Person E':[3,3,4,2]},
#     9:  {'Person A':[5,5,5,5],'Person B':[5,5,5,5],'Person C':[5,5,5,5],'Person D':[5,4,5,5],'Person E':[5,5,5,5]},
#     11: {'Person A':[5,5,4,5],'Person B':[5,4,5,5],'Person C':[5,2,5,5],'Person D':[5,3,5,5],'Person E':[5,5,5,5]},
#     13: {'Person A':[5,5,5,5],'Person B':[5,2,4,3],'Person C':[5,2,5,5],'Person D':[5,4,4,5],'Person E':[5,2,4,3]},
#     15: {'Person A':[5,5,5,5],'Person B':[5,4,5,5],'Person C':[5,5,5,5],'Person D':[5,4,5,5],'Person E':[5,4,5,4]},
#     17: {'Person A':[5,5,3,5],'Person B':[5,5,4,3],'Person C':[5,5,5,5],'Person D':[5,5,3,5],'Person E':[5,3,4,3]},
#     19: {'Person A':[5,5,4,5],'Person B':[5,5,4,3],'Person C':[5,5,5,5],'Person D':[5,4,4,5],'Person E':[5,3,5,4]},
#     21: {'Person A':[5,5,5,5],'Person B':[5,4,5,5],'Person C':[5,5,5,5],'Person D':[5,5,3,5],'Person E':[4,5,5,4]},
#     23: {'Person A':[5,5,5,5],'Person B':[5,5,5,5],'Person C':[5,5,5,5],'Person D':[5,4,3,5],'Person E':[5,3,5,3]},
# }

# rows = []
# for alt_text_id, people in scores_per_chart.items():
#     for _person, scores in people.items():  # Name wird hier nicht gespeichert (Schema hat keine Person-Spalte)
#         rows.append((alt_text_id, scores[0], scores[1], scores[2], scores[3]))

# cur.executemany("""
#     INSERT INTO people_evaluation
#         (alt_text_id, score_neutrality, score_clarity, score_conciseness, score_preceived_completeness)
#     VALUES (?, ?, ?, ?, ?)
# """, rows)

# conn.commit()
# conn.close()

In [24]:
# conn = sqlite3.connect(DB_PATH)

# # Tabelle11: people_evaluation
# conn.execute("""
# CREATE TABLE IF NOT EXISTS people_evaluation (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     alt_text_id INTEGER NOT NULL,
#     score_neutrality INTEGER,
#     score_clarity INTEGER,
#     score_conciseness INTEGER,
#     score_preceived_completeness INTEGER,
#     FOREIGN KEY (alt_text_id) REFERENCES alt_text(id)
# );
# """)

In [25]:
# import pandas as pd
# import numpy as np

# kriterien_lst = ['score_neutrality', 'score_clarity', 'score_conciseness', 'score_preceived_completeness']

# # scores_per_chart wie von dir definiert …

# # ---- Dict -> DataFrame (MultiIndex: alt_text_id, person) ----
# rows = []
# for alt_text_id, person_dict in scores_per_chart.items():
#     for person, scores in person_dict.items():
#         rows.append({
#             "alt_text_id": alt_text_id,
#             "person": person,
#             **{k: v for k, v in zip(kriterien_lst, scores)}
#         })

# df = pd.DataFrame(rows).set_index(["alt_text_id", "person"]).sort_index()

# # ---- Mittelwerte pro alt_text_id je Kriterium + Gesamtscore ----
# mean_per_alt_by_crit = df.groupby(level="alt_text_id")[kriterien_lst].mean()
# mean_per_alt_by_crit["overall_mean"] = mean_per_alt_by_crit.mean(axis=1)

# # Optional: Rangliste (höchster Gesamtscore zuerst)
# ranking = mean_per_alt_by_crit.sort_values("overall_mean", ascending=False)

# # ---- Optional: Mittelwerte pro Person (über alle alt_text_id) ----
# mean_per_person = df.groupby(level="person")[kriterien_lst].mean()
# mean_per_person["overall_mean"] = mean_per_person.mean(axis=1)
# mean_per_person = mean_per_person.sort_values("overall_mean", ascending=False)

# # ---- Beispiele für Ausgabe ----
# print("Mittelwerte je alt_text_id:")
# print(mean_per_alt_by_crit.round(3))
# print("\nRanking (overall_mean):")
# print(ranking["overall_mean"].round(3))
# print("\nMittelwerte je Person:")
# print(mean_per_person.round(3))


## Create alt_text_similarity_to_gold_standard Table

In [26]:
# # Tabelle15: alt_text_similarity_to_gold_standard
# conn.execute("""
# CREATE TABLE IF NOT EXISTS alt_text_similarity_to_gold_standard (
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     alt_text_id INTEGER NOT NULL,
#     similarity_score FLOAT,
#     FOREIGN KEY (alt_text_id) REFERENCES alt_text(id)
# );
# """)

## Checke alle Tabellen in DB mit:

In [27]:
DB_PATH = "chart_database.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# alle Tabellen holen (keine internen sqlite_* Tabellen)
cursor.execute("""
    SELECT name 
    FROM sqlite_master 
    WHERE type='table'
    AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
""")

tables = [row[0] for row in cursor.fetchall()]

print(f"Found {len(tables)} tables:\n")

for table in tables:
    cursor.execute(f"PRAGMA table_info({table});")
    cols = cursor.fetchall()

    col_names = [c[1] for c in cols]   # Spaltenname ist in PRAGMA table_info(...): index 1

    print(f"{table} ({', '.join(col_names)})")


Found 15 tables:

alt_text (id, chart_id, generation_run_id, variant_no, short_description_metadata, short_description_overview, long_description)
alt_text_embedding (id, alt_text_id, model_id, dim, normalized, embedding, metadata_embedding, overview_embedding, long_description_embedding, normalized_parts)
alt_text_similarity_to_gold_standard (id, alt_text_id, similarity_score)
chart (id, chart_type_id, x_axis_type, y_axis_type, x_axis_min, x_axis_max, y_axis_min, y_axis_max, complex, title, subtitle, notes)
chart_data (id, chart_id, x_value, x_category, y_value, y_category, row_index, col_index, prognosis, highlighted)
chart_type (id, type)
data_event (id, chart_id, type, date, label)
evaluation_run (id, generation_run_id, model_id, num_evals, func_clarity_id, func_completeness_id, func_conciseness_id, func_preceived_completeness_id, func_neutrality_id, func_correctness_id)
generation_run (id, model_id, prompt_text_function_id, created_at, temperature, n_variants)
gold_standard_alt_te

In [28]:
DB_PATH = "chart_database.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Alle Tabellen holen (ohne SQLite-internes Zeugs)
cursor.execute("""
    SELECT name FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
""")
tables = [row[0] for row in cursor.fetchall()]

print(f"Found {len(tables)} tables:\n")

for table in tables:
    print(f"TABLE: {table}")

    # Spaltennamen holen
    cursor.execute(f"PRAGMA table_info({table});")
    cols = [col[1] for col in cursor.fetchall()]
    print("Columns:", cols)

    # Erste 3 Datensätze holen
    cursor.execute(f"SELECT * FROM {table} LIMIT 4;")

    rows = cursor.fetchall()

    if not rows:
        print("(no rows)")
    else:
        for r in rows:
            print(r)

    print("-" * 60)

conn.close()


Found 15 tables:

TABLE: alt_text
Columns: ['id', 'chart_id', 'generation_run_id', 'variant_no', 'short_description_metadata', 'short_description_overview', 'long_description']
(1, '006f6723bd41797d2bb1e3f65444757f', 1, 1, 'Liniendiagramm. Titel: SMI avanciert kräftig. Untertitel: Kursentwicklung in Punkten. Waagrechte Achse: Zeitraum Q1 2017 bis Q4 2017. Senkrechte Achse: Ausschnitt 8000 bis 9600 Punkte. Daten: Täglich Werte.', 'Das ganze Jahr 2017 ist durch einen deutlichen Aufwärtstrend des SMI gekennzeichnet, unterbrochen von Phasen erhöhter Volatilität und Konsolidierung, besonders zwischen dem zweiten und dritten Quartal, bevor der Anstieg im vierten Quartal fortgesetzt wird.', 'Der SMI beginnt das Jahr 2017 bei rund 8316 Punkten, sinkt kurzzeitig auf etwa 8229 (Mitte Januar) und steigt dann stetig. Im April wird die 8800-Punkte-Marke überschritten, worauf eine Phase volatiler Ausschläge folgt, mit einem Hoch von 9177 (Anfang August) und einem Tief von 8814 (Ende August). Ab Sept

## Alle Tabellen auf einen Blick

### Tabelle1: model
cursor.execute("""
CREATE TABLE IF NOT EXISTS model (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    company TEXT,
    model_series TEXT,
    URL TEXT
);
""")

### Tabelle2: chart_type
cursor.execute("""
CREATE TABLE IF NOT EXISTS chart_type (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT
);
""")

### Tabelle3: chart
cursor.execute("""
CREATE TABLE IF NOT EXISTS chart (
    id TEXT PRIMARY KEY,
    chart_type_id INTEGER,
    x_axis_type TEXT,
    y_axis_type TEXT,
    x_axis_min FLOAT,
    x_axis_max FLOAT,
    y_axis_min FLOAT,
    y_axis_max FLOAT,
    complex BOOL,
    title TEXT,
    subtitle TEXT,
    notes TEXT,
    FOREIGN KEY (chart_type_id) REFERENCES chart_type(id)
);
""")

### Tabelle4: chart_data
cursor.execute("""
CREATE TABLE IF NOT EXISTS chart_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    chart_id TEXT,
    x_value TEXT,
    x_category TEXT,
    y_value TEXT,
    y_category TEXT,
    row_index INTEGER,
    col_index INTEGER,
    prognosis BOOL,
    highlighted BOOL,
    FOREIGN KEY (chart_id) REFERENCES chart(id)
);
""")

### Tabelle5: data_event
cursor.execute("""
CREATE TABLE IF NOT EXISTS data_event (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    chart_id TEXT,
    type TEXT,
    date TEXT,
    label TEXT,
    FOREIGN KEY (chart_id) REFERENCES chart(id)
);
""")

### Tabelle6: prompt_text_function
cursor.execute("""
CREATE TABLE IF NOT EXISTS prompt_text_function (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    content TEXT
);
""")

### Tabelle7: generation_run
conn.execute("""
CREATE TABLE IF NOT EXISTS generation_run (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    model_id INTEGER,
    prompt_text_function_id INTEGER,
    created_at TEXT,
    temperature FLOAT,
    n_variants INTEGER,
    FOREIGN KEY (model_id) REFERENCES model(id),
    FOREIGN KEY (prompt_text_function_id) REFERENCES prompt_text_function(id)
);
""")

### Tabelle8: alt_text
# (Wichtig: vor alt_text_embedding anlegen, da FK darauf zeigt)
conn.execute("""
CREATE TABLE IF NOT EXISTS alt_text (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    chart_id TEXT,
    generation_run_id INTEGER,
    variant_no INTEGER,
    short_description_metadata TEXT,
    short_description_overview TEXT,
    long_description TEXT,
    FOREIGN KEY (generation_run_id) REFERENCES generation_run(id),
    FOREIGN KEY (chart_id) REFERENCES chart(id)
);
""")

### Tabelle9: alt_text_embedding
# Korrigiert: meta_embedding -> metadata_embedding (wie im "Found"-Schema)
conn.execute("""
CREATE TABLE IF NOT EXISTS alt_text_embedding (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    alt_text_id INTEGER,
    model_id INTEGER,
    dim INTEGER,
    normalized INTEGER,
    embedding BLOB,
    metadata_embedding BLOB,
    overview_embedding BLOB,
    long_description_embedding BLOB,
    normalized_parts INTEGER,
    FOREIGN KEY (alt_text_id) REFERENCES alt_text(id),
    FOREIGN KEY (model_id) REFERENCES model(id)
);
""")

### Tabelle10: gold_standard_alt_text
# Korrigiert/ergänzt: Spalten wie im "Found"-Schema + normalized_embeddings
conn.executescript("""
CREATE TABLE IF NOT EXISTS gold_standard_alt_text (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    chart_id TEXT NOT NULL,

    short_description_metadata TEXT,
    short_description_overview TEXT,
    long_description TEXT,

    embedding BLOB,
    embedding_model_id INTEGER,

    tokens_short_description_metadata INTEGER,
    tokens_short_description_overview INTEGER,
    tokens_long_description INTEGER,

    judge_model_id INTEGER NOT NULL,
    func_clarity_id INTEGER,
    func_completeness_id INTEGER,
    func_conciseness_id INTEGER,
    func_preceived_completeness_id INTEGER,
    func_neutrality_id INTEGER,
    func_correctness_id INTEGER,

    score_clarity INTEGER,
    reason_clarity TEXT,
    score_completeness INTEGER,
    reason_completeness TEXT,
    score_conciseness INTEGER,
    reason_conciseness TEXT,
    score_preceived_completeness INTEGER,
    reason_preceived_completeness TEXT,
    score_neutrality INTEGER,
    reason_neutrality TEXT,
    score_correctness INTEGER,
    reason_correctness TEXT,

    -- wie im "Found"-Schema:
    embedding_metadata BLOB,
    embedding_overview BLOB,
    embedding_long_description BLOB,
    normalized_embeddings INTEGER,

    FOREIGN KEY (embedding_model_id) REFERENCES model(id),
    FOREIGN KEY (chart_id) REFERENCES chart(id),

    FOREIGN KEY (judge_model_id) REFERENCES model(id),
    FOREIGN KEY (func_clarity_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_completeness_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_conciseness_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_preceived_completeness_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_neutrality_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_correctness_id) REFERENCES prompt_text_function(id)
);
""")

### Tabelle11: people_evaluation
conn.execute("""
CREATE TABLE IF NOT EXISTS people_evaluation (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    alt_text_id INTEGER NOT NULL,
    score_neutrality INTEGER,
    score_clarity INTEGER,
    score_conciseness INTEGER,
    score_preceived_completeness INTEGER,
    FOREIGN KEY (alt_text_id) REFERENCES alt_text(id)
);
""")

### Tabelle12: evaluation_run
conn.execute("""
CREATE TABLE IF NOT EXISTS evaluation_run (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    generation_run_id INTEGER NOT NULL,
    model_id INTEGER NOT NULL,
    num_evals INTEGER NOT NULL,
    func_clarity_id INTEGER,
    func_completeness_id INTEGER,
    func_conciseness_id INTEGER,
    func_preceived_completeness_id INTEGER,
    func_neutrality_id INTEGER,
    func_correctness_id INTEGER,
    FOREIGN KEY (generation_run_id) REFERENCES generation_run(id),
    FOREIGN KEY (model_id) REFERENCES model(id),
    FOREIGN KEY (func_clarity_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_completeness_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_conciseness_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_preceived_completeness_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_neutrality_id) REFERENCES prompt_text_function(id),
    FOREIGN KEY (func_correctness_id) REFERENCES prompt_text_function(id)
);
""")

### Tabelle13: llm_evaluation
# Korrigiert/ergänzt: reasoning_* Felder wie im "Found"-Schema + FK auf alt_text
conn.execute("""
CREATE TABLE IF NOT EXISTS llm_evaluation (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    evaluation_run_id INTEGER,
    alt_text_id INTEGER,
    no_eval INTEGER,

    score_clarity INTEGER,
    reasoning_clarity TEXT,

    score_completeness INTEGER,
    reasoning_completeness TEXT,

    score_conciseness INTEGER,
    reasoning_conciseness TEXT,

    score_preceived_completeness INTEGER,
    reasoning_preceived_completeness TEXT,

    score_neutrality INTEGER,
    reasoning_neutrality TEXT,

    score_correctness INTEGER,
    reasoning_correctness TEXT,

    FOREIGN KEY (evaluation_run_id) REFERENCES evaluation_run(id),
    FOREIGN KEY (alt_text_id) REFERENCES alt_text(id)
);
""")

### Tabelle14: metric
conn.execute("""
CREATE TABLE IF NOT EXISTS metric (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    alt_text_id INTEGER NOT NULL,
    tokens_short_description_metadata INTEGER,
    tokens_short_description_overview INTEGER,
    tokens_long_description INTEGER,
    FOREIGN KEY (alt_text_id) REFERENCES alt_text(id)
);
""")

### Tabelle15: alt_text_similarity_to_gold_standard
conn.execute("""
CREATE TABLE IF NOT EXISTS alt_text_similarity_to_gold_standard (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    alt_text_id INTEGER NOT NULL,
    similarity_score FLOAT,
    FOREIGN KEY (alt_text_id) REFERENCES alt_text(id)
);
""")


In [38]:
DB_PATH = "chart_database.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    SELECT 
        evaluation_run_id,
        COUNT(*) AS anzahl_zeilen
    FROM llm_evaluation
    GROUP BY evaluation_run_id
    ORDER BY evaluation_run_id;
""")
for row in cursor.fetchall():
    print(row)
conn.close()


(6, 270)
(8, 270)
(9, 270)
(11, 12)


In [37]:
DB_PATH = "chart_database.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()


cursor.execute("""
DELETE FROM llm_evaluation
WHERE evaluation_run_id = ?
""", (5,))
conn.commit()
conn.close()